# 04 Ridge Regression Models

Trains Ridge regression models for PM2.5, PM10, and NO₂ rate prediction.

**Run order:** this notebook is part of the AirAware workflow and uses the shared repository folders: `data/`, `data/processed/`, and `outputs/`.


# Train 3 Linear Regression Models for Indoor IAQ Pollutants
# Targets: indoor_pm25, indoor_pm10, indoor_no2


In [ ]:
#import libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, RidgeCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from pathlib import Path


In [ ]:
# ============================================================
# Project paths
# ============================================================
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

DATA_DIR.mkdir(exist_ok=True)
PROCESSED_DATA_DIR.mkdir(exist_ok=True)
OUTPUTS_DIR.mkdir(exist_ok=True)


In [ ]:
# ------------------------------------------------------------
# 1. Load dataset
# ------------------------------------------------------------

df = pd.read_csv(PROCESSED_DATA_DIR / "air_quality_cleaned_featured.csv")
dftest = pd.read_csv(PROCESSED_DATA_DIR / "air_quality_model_ready_encoded.csv")

In [ ]:
# ------------------------------------------------------------
# 2. Create interaction features
# ------------------------------------------------------------

df["cooking_x_window"] = df["cooking_event"] * df["window_open"]
df["cooking_x_fan"] = df["cooking_event"] * df["extractor_fan_on"]
df["gas_cooking_x_fan"] = df["gas_cooking_or_gas_hob"] * df["extractor_fan_on"]

In [ ]:
# ------------------------------------------------------------
# Remove duplicate / redundant interaction features
# ------------------------------------------------------------

duplicate_interaction_cols = [
    "gas_cooking_x_extractor_fan",
    "cooking_x_window_open"
]

duplicate_interaction_cols = [
    col for col in duplicate_interaction_cols if col in df.columns
]

df = df.drop(columns=duplicate_interaction_cols)

print("Dropped duplicate interaction columns:", duplicate_interaction_cols)

In [ ]:
# ------------------------------------------------------------
# 3. Define model features and targets
# ------------------------------------------------------------

targets = {
    "PM2.5": "indoor_pm25",
    "PM10": "indoor_pm10",
    "NO2": "indoor_no2"
}

leakage_cols = [
    "pm25_rate", "pm10_rate", "no2_rate",
    "cooking_pm25_raw", "cooking_pm10_raw", "cooking_no2_raw",
    "vent_pm25_multiplier", "vent_pm10_multiplier", "vent_no2_multiplier",
    "pm25_infiltration_factor", "pm10_infiltration_factor", "no2_infiltration_factor",
    "indoor_pm25", "indoor_pm10", "indoor_no2"
]

datetime_cols = [
    "day",
    "start_time_local",
    "end_time_local"
]

cols_to_drop = [col for col in leakage_cols + datetime_cols if col in df.columns]

X = df.drop(columns=cols_to_drop)

# Identify feature types
categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_features = X.select_dtypes(include=["int64", "float64", "bool"]).columns.tolist()

print("Categorical features:", categorical_features)
print("Numeric features:", numeric_features)

In [ ]:
# ------------------------------------------------------------
# 4. Preprocessing
# ------------------------------------------------------------

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_features)
    ]
)


In [ ]:
# ------------------------------------------------------------
# 5. Train and evaluate models
# ------------------------------------------------------------
alphas = np.logspace(-3, 3, 50)
results = {}
coefficient_tables = {}

for pollutant_name, target_col in targets.items():

    y = df[target_col]

    X_train_ridgecv, X_test_ridgecv, y_train_ridgecv, y_test_ridgecv = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42
    )

    model = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("regressor", RidgeCV(alphas=alphas))
        ]
    )


    model.fit(X_train_ridgecv, y_train_ridgecv)

    y_pred_ridgecv = model.predict(X_test_ridgecv)

    r2 = r2_score(y_test_ridgecv, y_pred_ridgecv)
    mae = mean_absolute_error(y_test_ridgecv, y_pred_ridgecv)
    rmse = np.sqrt(mean_squared_error(y_test_ridgecv, y_pred_ridgecv))

    results[pollutant_name] = {
        "Target": target_col,
        "R2": r2,
        "MAE": mae,
        "RMSE": rmse
    }

    # --------------------------------------------------------
    # Extract coefficients
    # --------------------------------------------------------

    fitted_preprocessor = model.named_steps["preprocessor"]
    regressor = model.named_steps["regressor"]

    feature_names = fitted_preprocessor.get_feature_names_out()

    coef_df = pd.DataFrame({
        "feature": feature_names,
        "coefficient": regressor.coef_
    })

    coef_df["absolute_coefficient"] = coef_df["coefficient"].abs()

    coef_df = coef_df.sort_values(
        by="absolute_coefficient",
        ascending=False
    )

    coefficient_tables[pollutant_name] = coef_df

In [ ]:
# ------------------------------------------------------------
# 6. Print model performance
# ------------------------------------------------------------

results_df = pd.DataFrame(results).T
print("\nModel Performance:")
print(results_df)


In [ ]:
# ------------------------------------------------------------
# 7. Print top coefficients for each model
# ------------------------------------------------------------

for pollutant_name, coef_df in coefficient_tables.items():
    print(f"\nTop coefficients for {pollutant_name} model:")
    print(coef_df.head(15)[["feature", "coefficient"]])


In [ ]:
# ------------------------------------------------------------
# 8. Save outputs
# ------------------------------------------------------------
OUT_DIR   = OUTPUTS_DIR / "ridge_regression"
OUT_DIR.mkdir(exist_ok=True)
results_df.to_csv(OUT_DIR / "ridgecv_regression_model_performance.csv", index=False)

for pollutant_name, coef_df in coefficient_tables.items():
    safe_name = pollutant_name.replace(".", "").replace(" ", "_").lower()
    coef_df.to_csv(OUT_DIR / f"{safe_name}_ridgecv_regression_coefficients.csv", index=False)

In [ ]:
#create new dataframe to store scores
dfscore = pd.DataFrame()


In [ ]:
# WHO-style reference thresholds in µg/m³
THRESHOLDS = {
    "pm25": 15,   # 24-hour PM2.5 guideline
    "pm10": 45,   # 24-hour PM10 guideline
    "no2": 25     # 24-hour NO2 guideline
}

WEIGHTS = {
    "pm25": 0.40,
    "pm10": 0.30,
    "no2": 0.30
}

df["pm25_score"] = df["indoor_pm25"] / THRESHOLDS["pm25"]
df["pm10_score"] = df["indoor_pm10"] / THRESHOLDS["pm10"]
df["no2_score"] = df["indoor_no2"] / THRESHOLDS["no2"]

df["weighted_iaq_score"] = (
    WEIGHTS["pm25"] * df["pm25_score"] +
    WEIGHTS["pm10"] * df["pm10_score"] +
    WEIGHTS["no2"] * df["no2_score"]
)

In [ ]:
def classify_iaq(score):
    if score < 1:
        return "Good"
    elif score < 2:
        return "Moderate"
    elif score < 3:
        return "Poor"
    else:
        return "Very Poor"

df["iaq_category"] = df["weighted_iaq_score"].apply(classify_iaq)

In [ ]:
#export the dataframe with scores and categories
df.to_csv(OUT_DIR / "ridge_regression_indoor_air_quality_scores.csv", index=False)